## Utilisation d'un PINN comme méthode d'inversion

In [1]:
import jax
import jax.numpy as jnp
import functools
import numpy as np
import optax
import matplotlib.pyplot as plt
from data_load import WaveguideBoundaryData

In [2]:
print(jax.devices())

[CpuDevice(id=0)]


**Data loading**

In [ ]:
#Domaine Omega : [-1,1]x[0,0.6]
#conditions de neuman en y=0 et y=0.6
H = 0.6

data_loader_left = WaveguideBoundaryData('data/pinn_boundary_left_n0.csv')
data_loader_right = WaveguideBoundaryData('data/pinn_boundary_right_n0.csv')

X_left, Y_left, U_re_left, U_im_left, freq = data_loader_left.get_training_data()
X_right, Y_right, U_re_right, U_im_right, _ = data_loader_right.get_training_data()

print(Y_left[freq[0]])
print(freq)

[0.58 0.6  0.56 0.54 0.52 0.5  0.48 0.46 0.44 0.42 0.4  0.38 0.36 0.34
 0.32 0.3  0.28 0.26 0.24 0.22 0.2  0.18 0.16 0.14 0.12 0.1  0.02 0.04
 0.08 0.06]
['200' '400' '600' '700' '1000' '1200' '1400' '1500']


**Normalisation des outputs et inputs**

In [ ]:
# x est déjà dans [-1,1]
#on passe y et u au bord dans [-1,1]

for f in freq:
    U_norm = np.sqrt(max(np.concatenate((U_re_left[f]**2 + U_im_left[f]**2,U_re_right[f]**2 + U_im_right[f]**2))))
    U_re_left[f]/=U_norm
    U_im_left[f]/=U_norm
    U_re_right[f]/=U_norm
    U_im_right[f]/=U_norm
    Y_norm = max(Y_left[f])
    Y_left[f]=2*Y_left[f]/Y_norm - 1
    Y_right[f]=2*Y_left[f]/Y_norm - 1

**Initialisation des paramètres**

In [ ]:
key = jax.random.key(0) # initialisation de la PRNG

#Initialisation des Fouriers Features
#On connait notre kmax car c0 = 340
fmax = max(freq.astype(np.float32))
c0 = 340
contrast = 1 #pour l'instant
kmax = 2*jnp.pi*fmax*contrast/340
m = 10

key, subkey = jax.random.split(key)
B = jax.random.uniform(subkey, (m, 2), minval=-kmax, maxval=kmax)

def gamma(x,y):
    v = jnp.array([x,y])
    return jnp.concatenate(jnp.cos(2*np.pi*B@v),jnp.sin(2*np.pi*B@v))

gamma_vmap = jax.vmap(gamma, (0,0))

#Il faut initialiser les points de quadrature et les cn

N_modes = jnp.round(2*H*fmax/c0) + 5

def c_n(y,n):
    return jnp.sqrt(2/H)*jnp.cos(n*np.pi*y/H)

c_n_vmap = jax.vmap(c_n, (0,None))

y_gauss_legendre, w_gauss_legendre = jnp.polynomial.legendre.leggauss(20)

# Entrée (x, y)
n_input = 2

n_layers_uv = [n_input*2*m, 64, 64, 64, 2]
n_layers_c = [n_input, 64, 1]

#Initialisation de Xavier

def init_layers(key, n_layers):
    layers = []
    for i in range(len(n_layers)-1):
        key, subkey = jax.random.split(key)
        W = jax.random.normal(subkey, (n_layers[i], n_layers[i+1])) / jnp.sqrt(n_layers[i])
        b = jnp.zeros(n_layers[i+1])
        layers.append({"W": W, "b": b})
    return layers, key

key, subkey_uv, subkey_c = jax.random.split(key, 3)

layers_uv, key = init_layers(subkey_uv, n_layers_uv)
layers_c, key = init_layers(subkey_c, n_layers_c)

# On regroupe les 3 réseaux dans le dictionnaire params
params = {
    "layers_uv": layers_uv,
    "layers_c": layers_c,
}

**Le Forward**

In [ ]:
def forward_func(layers,X):
    n = len(layers)
    Z = X
    for i in range(n-1):
        Z = jax.nn.tanh(Z @ layers[i]["W"] + layers[i]["b"])
    Z = Z @ layers[-1]["W"] + layers[-1]["b"]
    return Z

def forward_params(layers,X):
    n = len(layers)
    Z = X
    for i in range(n-1):
        Z = jax.nn.tanh(Z @ layers[i]["W"] + layers[i]["b"])
    Z =jax.nn.gelu(Z @ layers[-1]["W"] + layers[-1]["b"])
    return Z

**La loss function**

In [ ]:
def loss_fn(params, x, y, f):

    def uv(x,y):
        return forward_func(params["layers_uv"], jnp.array([x, y]))
    
    def k(x,y):
        return 2*jnp.pi*f/forward_params(params["layers_c"], jnp.array([x, y]))

    def pde_residual(x, y):
        uv_xx = jax.grad(jax.grad(uv, argnums=0), argnums=0)(x, y)
        uv_yy = jax.grad(jax.grad(uv, argnums=1), argnums=1)(x, y)
        return uv_xx + uv_yy + k(x,y)**2*uv
    
    def DtN(uv, )
    
    vmap_uv = jax.vmap(uv, in_axes=(0,0))
    vmap_pde_residual = jax.vmap(pde_residual, in_axes=(0,0))

    pde_loss = jnp.mean(vmap_pde_residual(x,y)**2)

    total_loss = pde_loss + bc_loss
    
    return total_loss, (pde_loss, bc_loss)

**Le training**

**La phase d'entrainement**

In [ ]:
num_steps = 50000
eval_interval = num_steps//20
Nmc = 5000

key, subkey = jax.random.split(key)
best_params, key, train_loss, test_loss = train(params, Nmc, num_steps, eval_interval, subkey)

In [ ]:
#Print des métriques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.semilogy(eval_interval*np.arange(num_steps//eval_interval), np.array(train_loss)[:,0], label="PDE Loss")
ax1.semilogy(eval_interval*np.arange(num_steps//eval_interval), np.array(train_loss)[:,1], label="BC Loss")
ax1.semilogy(eval_interval*np.arange(num_steps//eval_interval), np.array(train_loss)[:,2], label="IC Loss")

ax2.semilogy(eval_interval*np.arange(num_steps//eval_interval), test_loss, label="$L^2$ relative error")

ax1.set_title("Train Loss")
ax1.set_xlabel("steps")
ax1.set_ylabel("Loss")
ax1.legend(loc="upper right")
ax1.grid(True)

ax2.set_title("Test Loss")
ax2.set_xlabel("steps")
ax2.legend(loc="upper right")
ax2.grid(True)

plt.tight_layout()
plt.savefig('fig/Heat_eq_1D_Loss_base_f_50.pdf')
plt.show()